In [25]:
import sys
import os
import numpy as np
import tensorflow as tf
from pydub import AudioSegment

# Ajouter src au path pour importer feature_extraction
sys.path.append(os.path.abspath("../src"))
from feature_extraction import extract_mfcc

# -----------------------------
# Configuration & Chargement
# -----------------------------
MODEL_PATH = "../src/speech_model.keras"
MAPPING_PATH = "../data/processed/class_mapping.npy"
TRAIN_DATA_PATH = "../data/processed/X_train.npy"  # Pour récupérer mean/std

# Charger le modèle
model = tf.keras.models.load_model(MODEL_PATH)

# Charger le mapping des classes
class_mapping = np.load(MAPPING_PATH, allow_pickle=True).item()
index_to_class = {v: k for k, v in class_mapping.items()}

# Calculer les stats de normalisation comme dans data_utils.py
X_train_sample = np.load(TRAIN_DATA_PATH)
MEAN = np.mean(X_train_sample)
STD = np.std(X_train_sample)
if STD == 0:
    STD = 1e-8

# -----------------------------
# Conversion M4A/MP3 -> WAV 16kHz mono
# -----------------------------
def convert_to_wav(input_file, output_file="tmp_audio.wav"):
    audio = AudioSegment.from_file(input_file)
    audio = audio.set_channels(1).set_frame_rate(16000)
    audio.export(output_file, format="wav")
    return output_file

# -----------------------------
# Fonction de prédiction
# -----------------------------
def predict_audio(file_path):
    tmp_file = None

    # Conversion si nécessaire
    if not file_path.lower().endswith(".wav"):
        tmp_file = "tmp_audio.wav"
        file_path = convert_to_wav(file_path, tmp_file)

    # Extraction MFCC
    mfcc = extract_mfcc(file_path, n_mfcc=40, max_len=44)

    # Normalisation (avec MEAN et STD du train set)
    mfcc = (mfcc - MEAN) / STD

    # Préparer pour le CNN
    mfcc = mfcc[..., np.newaxis]      # (40,44,1)
    mfcc = np.expand_dims(mfcc, axis=0)  # (1,40,44,1)

    # Prédiction
    prediction = model.predict(mfcc, verbose=0)
    predicted_index = np.argmax(prediction)
    predicted_word = index_to_class[predicted_index]
    confidence = np.max(prediction)

    print(f"\n--- Résultat ---")
    print(f"Mot prédit : {predicted_word}")
    print(f"Confiance  : {confidence*100:.2f}%")

    # Nettoyer fichier temporaire
    if tmp_file and os.path.exists(tmp_file):
        os.remove(tmp_file)

    return predicted_word



In [58]:
audio_file = "../data/augmentation/one.mp3"  # ou .wav / .mp3
a= predict_audio(audio_file)
print(a)


--- Résultat ---
Mot prédit : one
Confiance  : 100.00%
one
